### 00_data_reprocessing
* Setup python venv environment
* Installs dependencies
* Checks dataset files are present 
    * Prints how many files in each dataset

Possible addition (future):
* image alignment cropping


In [10]:
# ensure dependencies are installed in the current Python environment
import sys
from pathlib import Path
requirements_path = Path("../requirements.txt")
!{sys.executable} -m pip install -q -r {requirements_path}

In [11]:
# Config + imports
import os
from pathlib import Path
import hashlib
import json
from PIL import Image
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import imagehash
import cv2
import torch

# Paths: update these if needed
ROOT_DIR = Path("../data_raw")         # where your datasets live
OUT_DIR = Path("../data_processed")    # where processed outputs will go
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Run options
SAMPLE_LIMIT = None   # set to an int to process fewer images during testing
MIN_FACE_SIDE = 80    # min face side in pixels to accept (tweak for your data)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
repo_root = ROOT_DIR.parent.resolve()

def to_rel(p: Path) -> str:
    """Return a stable relative path string from repo_root, fallback to name."""
    try:
        return str(p.relative_to(repo_root).as_posix())
    except Exception:
        try:
            return str(p.relative_to(Path.cwd()).as_posix())
        except Exception:
            return str(p.name)

Device: cuda


In [4]:
# Check: counts the number of files on disk (in this case, images)

from pathlib import Path
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pgm"}   
vid_exts = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv", ".mpg", ".mpeg"}
exts = img_exts | vid_exts
ROOT = Path("../data_raw").resolve()   # same ROOT_DIR used; adjust if needed
print("Scanning disk under:", to_rel(ROOT))
if not ROOT.exists():
    raise FileNotFoundError(f"{ROOT} not found from this kernel. Check ROOT_DIR.")

files = [p for p in ROOT.rglob("*") if p.is_file() and p.suffix.lower() in exts]
print("Total media files on disk:", len(files))

from collections import Counter
tops = [p.relative_to(ROOT).parts[0] if len(p.relative_to(ROOT).parts) else "" for p in files]
for k,v in Counter(tops).most_common():
    print(f"  {k}: {v}")

Scanning disk under: data_raw
Total media files on disk: 198394
  vggface2: 197693
  ibeta1: 701
